# P2-A3 — Embeddings & Semantic Search

This is the payoff moment. In **W2-A1** you built cosine similarity by hand on *fake* vectors. Now you'll get **real embeddings** from OpenAI and run **semantic search** — find the most relevant document for a query *by meaning*, not by keyword. This is the 'R' (Retrieval) in RAG, which you'll build next.

Goal: see that two texts about the same idea land close together in vector space, even with zero shared words.

In [1]:
# --- Setup (given) ---
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
EMBED_MODEL = 'text-embedding-3-small'   # cheap, 1536-dim embeddings

def embed(texts):
    """Embed a string or list of strings. Returns a (n, dim) numpy array."""
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

# --- Given: a tiny knowledge base + a query phrased with DIFFERENT words ---
docs = [
    "Our return policy allows refunds within 30 days of purchase.",
    "The mitochondria is the powerhouse of the cell.",
    "You can reset your password from the account settings page.",
    "Python list comprehensions make iterating over data concise.",
]
query = "How do I get my money back?"   # note: shares NO keywords with the refund doc
print('docs:', len(docs))

docs: 4


## Task 1 — Embed something and look at it
Embed the `query` and print: the **shape** of the vector, and its first 5 numbers.
<br>Goal: build intuition that text → a fixed-length list of floats. How many dimensions does `text-embedding-3-small` give you?

In [2]:
# TODO: embed the query; print shape and first 5 values

# embed the query; print shape and first 5 values
query_emb = embed(query)          # shape (1, 1536)
print('shape:', query_emb.shape)
print('first 5 values:', query_emb[0][:5])
print('dimensions:', query_emb.shape[1])   # text-embedding-3-small → 1536


shape: (1, 1536)
first 5 values: [-0.00460434 -0.01728821 -0.01272583  0.04129028 -0.05560303]
dimensions: 1536


## Task 2 — Semantic search: rank docs by similarity to the query
Embed all `docs` and the `query`, then compute cosine similarity of the query against every doc and rank them. Print which doc is the best match.
<br>Reuse your **W2-A1** approach: `docs_emb @ query_emb` for dot products, `np.linalg.norm(..., axis=1)` for magnitudes. The best match should be the refund doc — **even though the query says "money back", not "refund"**.
<br>Hint: bring over your `cosine_similarity` logic. `np.argsort(sims)[::-1]` to rank.

In [3]:
# TODO: embed docs + query, cosine-similarity rank, print best match

# embed docs + query, cosine-similarity rank, print best match
docs_emb = embed(docs)            # (4, 1536)
q = embed(query)[0]               # (1536,)

def cosine_similarity(matrix, vector):
    dots = matrix @ vector
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(vector)
    return dots / norms

sims = cosine_similarity(docs_emb, q)
ranking = np.argsort(sims)[::-1]   # high → low

print('Ranked matches:')
for rank, i in enumerate(ranking, 1):
    print(f'  {rank}. ({sims[i]:.3f}) {docs[i]}')

print('\nBest match:', docs[ranking[0]])


Ranked matches:
  1. (0.301) Our return policy allows refunds within 30 days of purchase.
  2. (0.183) You can reset your password from the account settings page.
  3. (-0.059) Python list comprehensions make iterating over data concise.
  4. (-0.061) The mitochondria is the powerhouse of the cell.

Best match: Our return policy allows refunds within 30 days of purchase.


## Task 3 — Prove it beats keyword search
Write a naive keyword search: does any doc literally contain words from the query (e.g. 'money', 'back')? Show it **fails** to find the refund doc, while your semantic search **succeeds**.
<br>Goal: this single contrast is the whole reason embeddings exist. Make it concrete.

In [4]:
# TODO: naive keyword match (fails) vs your semantic match (succeeds)

# naive keyword match (fails) vs your semantic match (succeeds)
query_words = {'money', 'back'}   # the meaningful words in the query

print('--- Keyword search ---')
keyword_hits = [d for d in docs if query_words & set(d.lower().replace('.', '').split())]
if keyword_hits:
    print('Found:', keyword_hits)
else:
    print('No doc contains the words', query_words, '-> keyword search FAILS')

print('\n--- Semantic search ---')
print('Found:', docs[ranking[0]], '-> SUCCEEDS (matches by meaning, not words)')


--- Keyword search ---
No doc contains the words {'back', 'money'} -> keyword search FAILS

--- Semantic search ---
Found: Our return policy allows refunds within 30 days of purchase. -> SUCCEEDS (matches by meaning, not words)


## Task 4 — Explain (in your own words)
1. What does an embedding actually *capture* about a piece of text, such that 'money back' and 'refund' end up close together?
2. Sketch (in words) how you'd use this to answer a user's question over a large document set — i.e. the retrieval half of RAG. What are the steps?

> _your answer here_
1. An embedding maps text into a high-dimensional vector (1536 dims here) that captures its meaning — the semantic relationships learned from training on huge amounts of text — rather than its literal characters. "Money back" and "refund" share almost no letters, but they're used in the same contexts (returns, purchases, getting paid), so the model places their vectors close together. Distance in this space ≈ difference in meaning, which is why cosine similarity surfaces the refund doc despite zero shared keywords.

2. The retrieval half of RAG: (1) Index ahead of time — split the document set into chunks, embed every chunk once, and store the vectors (with their source text) in a vector store/database. (2) At query time — embed the user's question with the same model. (3) Search — compute similarity between the query vector and all chunk vectors (cosine / nearest-neighbor) and take the top-k most similar chunks. (4) Return those chunks as the relevant context. That retrieved context is then fed alongside the question into the LLM (the "generation" half) so it answers grounded in your actual documents instead of guessing.